# 04 · Qualitative results: predictions of each model on the other biome

This notebook draws example predictions for the poster and the paper. It needs the checkpoints written by `02_train.ipynb`
(`results/raw/ckpt/`, not in git) and the pretrained encoder weights, because a checkpoint holds only the trained parameters.

**No new numbers.** All metrics in the paper come from `03_analysis.ipynb`. This notebook only shows pictures, and the per-image
mIoU it prints is a check that the loaded checkpoints behave as in training (it must match `results/raw/per_image/`).

**How images are chosen (fixed rule, no cherry-picking).** For one seed and one training biome, every model predicts every
official test image of the other biome. Images are sorted by their mIoU averaged over the models, and we show the images at the
10th, 50th and 90th percentile: a hard, a typical and an easy image.

**Code reuse.** The data loading, model and prediction code is taken from `02_train.ipynb` itself (sections 1–4 are executed
here, section 5, which trains, is not), so there is one copy of that code.

Settings (environment variables): the same as `02_train.ipynb` (`DATA_ROOT`, `RESULTS_DIR`, `DINOV2_NAME`, `DINOV3*_NAME`), plus
`QUAL_SEED` (default 0), `QUAL_TRAIN_BIOME` (default `amazon`) and `QUAL_MODELS` (default: all models with a checkpoint).

In [ ]:
import json, os
from pathlib import Path

os.environ.setdefault("SEEDS", os.environ.get("QUAL_SEED", "0"))
nb = json.loads(Path("02_train.ipynb").read_text())
for cell in nb["cells"]:
    src = "".join(cell["source"])
    if cell["cell_type"] == "markdown" and src.lstrip().startswith("## 5"):
        break                                     # section 5 runs the training: stop before it
    if cell["cell_type"] == "code":
        exec(compile(src, "02_train.ipynb", "exec"), globals())

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from safetensors.torch import load_file

SEED = int(os.environ.get("QUAL_SEED", "0"))
TRAIN = os.environ.get("QUAL_TRAIN_BIOME", "amazon")
OTHER = next(b for b in BIOMES if b != TRAIN)
FIGS = CONFIG["results_dir"] / "figures"; FIGS.mkdir(parents=True, exist_ok=True)
TABLES = CONFIG["results_dir"] / "tables"; TABLES.mkdir(parents=True, exist_ok=True)
# only complete runs: the runs/ CSV is written last, after the checkpoint and the per-image results
models = [k for k in env_list("QUAL_MODELS", ",".join(ALL_MODELS))
          if (RAW / "ckpt" / f"{run_id(k, SEED, TRAIN)}.safetensors").exists() and (RAW / "runs" / f"{run_id(k, SEED, TRAIN)}.csv").exists()]
print("complete runs with a checkpoint:", models)

mean, std = norm_stats(TRAIN)
test = DATA[(OTHER, "test")]
preds, rows = {}, []
for kind in models:
    model = build_model(kind)
    state = load_file(str(RAW / "ckpt" / f"{run_id(kind, SEED, TRAIN)}.safetensors"))
    missing, unexpected = model.load_state_dict(state, strict=False)
    assert not unexpected, unexpected
    # everything that is not in the checkpoint must be a frozen pretrained parameter
    trained = {n for n, p in model.named_parameters() if p.requires_grad} | {n for n, _ in model.named_buffers()}
    assert not (set(missing) & trained), sorted(set(missing) & trained)[:5]
    model.to(DEVICE)
    with torch.no_grad():
        p, _ = predict(model, test["x"], mean, std)
    preds[kind] = p
    c = confusion(p, test["m"])
    for name, ci in zip(test["names"], c):
        rows.append(dict(model=kind, image=name, **scores(ci[None])))
    print(f"{MODEL_NAMES[kind]:38s} pooled mIoU on {OTHER} test: {scores(c)['miou']:.4f}")
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

per_image = pd.DataFrame(rows)

Check: the per-image confusion counts must match those that `02_train.ipynb` wrote during training (same checkpoint, same
images). Small differences are possible only if this notebook runs on another device type (e.g. Apple MPS instead of CUDA).

In [ ]:
for kind in models:
    ref = pd.read_csv(RAW / "per_image" / f"{run_id(kind, SEED, TRAIN)}.csv")
    ref = ref[ref["eval_set"] == f"{OTHER}_test"].set_index("image")
    c = confusion(preds[kind], test["m"])
    diff = np.abs(c - ref.loc[test["names"], ["tp", "fp", "fn", "tn"]].to_numpy()).sum() / c.sum()
    print(f"{kind:14s} share of pixels that differ from the training-time evaluation: {diff:.2e}")

In [ ]:
order = per_image.groupby("image")["miou"].mean().sort_values()
pick = [order.index[int(round(q * (len(order) - 1)))] for q in (0.1, 0.5, 0.9)]
per_image.pivot(index="image", columns="model", values="miou").loc[order.index].round(4).to_csv(
    TABLES / f"qualitative_per_image_{TRAIN}_to_{OTHER}_s{SEED}.csv")
print("chosen images (10th, 50th, 90th percentile of the mean mIoU):", pick)

In [ ]:
def rgb(x):
    """True-colour image with a 2-98 % percentile stretch per band."""
    v = x[:3].astype(np.float32)
    lo, hi = np.percentile(v, 2, axis=(1, 2), keepdims=True), np.percentile(v, 98, axis=(1, 2), keepdims=True)
    return np.clip((v - lo) / np.maximum(hi - lo, 1), 0, 1).transpose(1, 2, 0)

# 0 correct non-forest, 1 correct forest, 2 false forest (FP), 3 missed forest (FN)
ERR_CMAP = ListedColormap(["#efe9d8", "#2e7d32", "#e8590c", "#1971c2"])
cols = ["image", "reference"] + models
fig, axes = plt.subplots(len(pick), len(cols), figsize=(2.1 * len(cols), 2.25 * len(pick)), squeeze=False)
for r, name in enumerate(pick):
    i = test["names"].index(name)
    t = test["m"][i].astype(bool)
    axes[r, 0].imshow(rgb(test["x"][i]))
    axes[r, 1].imshow(t.astype(int), cmap=ERR_CMAP, vmin=0, vmax=3, interpolation="nearest")
    axes[r, 0].set_ylabel(f"{['hard', 'typical', 'easy'][r]}\n{name[-5:]}", fontsize=8)
    for j, kind in enumerate(models):
        p = preds[kind][i]
        err = np.where(p & t, 1, np.where(p & ~t, 2, np.where(~p & t, 3, 0)))
        ax = axes[r, 2 + j]
        ax.imshow(err, cmap=ERR_CMAP, vmin=0, vmax=3, interpolation="nearest")
        m = per_image[(per_image.model == kind) & (per_image.image == name)]["miou"].item()
        ax.text(0.03, 0.97, f"{m:.2f}", transform=ax.transAxes, va="top", fontsize=8,
                bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.8))
for j, c in enumerate(cols):
    axes[0, j].set_title(c if j < 2 else MODEL_NAMES[c].replace(", ", "\n"), fontsize=8)
for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])
handles = [plt.Rectangle((0, 0), 1, 1, color=ERR_CMAP(k)) for k in range(4)]
fig.legend(handles, ["non-forest (correct)", "forest (correct)", "false forest", "missed forest"],
           loc="lower center", ncol=4, fontsize=8, frameon=False)
fig.suptitle(f"Trained on {BIOMES[TRAIN].title()}, tested on {BIOMES[OTHER].title()} (official test split, seed {SEED}; "
             f"number = image mIoU)", fontsize=9)
fig.tight_layout(rect=(0, 0.04, 1, 0.97))
out = FIGS / f"fig4_qualitative_{TRAIN}_to_{OTHER}_s{SEED}.png"
fig.savefig(out, dpi=200)
print("saved", out)

### Poster version

Fewer models (one per encoder family, plus satellite pretraining), two images (hard and typical) and fonts for an A0 poster column
(about 14 inches wide).

In [ ]:
POSTER_MODELS = [k for k in ["unet_ft", "dinov2_ft", "dinov3_ft", "dinov3l_pft", "dinov3sat_pft"] if k in preds]
POSTER_NAMES = {"unet_ft": "U-Net\nResNet-34", "dinov2_ft": "DINOv2\nViT-S", "dinov3_ft": "DINOv3\nViT-S",
                "dinov3l_pft": "DINOv3\nViT-L web", "dinov3sat_pft": "DINOv3\nViT-L sat"}
cols = ["image", "reference"] + POSTER_MODELS
fig, axes = plt.subplots(2, len(cols), figsize=(14, 5.6), squeeze=False)
for r, name in enumerate(pick[:2]):
    i = test["names"].index(name); t = test["m"][i].astype(bool)
    axes[r, 0].imshow(rgb(test["x"][i]))
    axes[r, 1].imshow(t.astype(int), cmap=ERR_CMAP, vmin=0, vmax=3, interpolation="nearest")
    axes[r, 0].set_ylabel(["hard", "typical"][r], fontsize=18)
    for j, kind in enumerate(POSTER_MODELS):
        p = preds[kind][i]
        err = np.where(p & t, 1, np.where(p & ~t, 2, np.where(~p & t, 3, 0)))
        ax = axes[r, 2 + j]; ax.imshow(err, cmap=ERR_CMAP, vmin=0, vmax=3, interpolation="nearest")
        m = per_image[(per_image.model == kind) & (per_image.image == name)]["miou"].item()
        ax.text(0.04, 0.96, f"{m:.2f}", transform=ax.transAxes, va="top", fontsize=15,
                bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.85))
for j, c in enumerate(cols):
    axes[0, j].set_title(POSTER_NAMES.get(c, c), fontsize=16)
for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])
fig.legend([plt.Rectangle((0, 0), 1, 1, color=ERR_CMAP(k)) for k in range(4)],
           ["non-forest", "forest", "false forest", "missed forest"], loc="lower center", ncol=4, fontsize=16, frameon=False)
fig.tight_layout(rect=(0, 0.08, 1, 1), w_pad=0.3, h_pad=0.3)
fig.savefig(FIGS / f"poster_qualitative_{TRAIN}_to_{OTHER}_s{SEED}.png", dpi=200)
print("saved poster figure")

## Method overview (poster figure)

One example image per biome, chosen by a fixed rule: the official test image whose forest fraction is the median of that split.

In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

def median_forest_image(b):
    f = DATA[(b, "test")]["m"].mean((1, 2))
    return int(np.argsort(f)[len(f) // 2])

def draw_method(out, k=1.0, dpi=250):
    """k < 1 draws a smaller canvas with the same font sizes, so the fonts are larger once the figure is scaled to its print width."""
    fig = plt.figure(figsize=(12 * k, 4.0 * k))
    ax = fig.add_axes((0, 0, 1, 1)); ax.set_xlim(0, 12); ax.set_ylim(0, 4.0); ax.axis("off")
    def box(x, y, w, h, text, fc, fs=8.5):
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.04,rounding_size=0.12", fc=fc, ec="#555", lw=0.8))
        ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=fs)
    def arrow(x0, y0, x1, y1, text="", color="#333"):
        ax.add_patch(FancyArrowPatch((x0, y0), (x1, y1), arrowstyle="-|>", mutation_scale=14, color=color, lw=1.5))
        if text:
            ax.text((x0 + x1) / 2, max(y0, y1) + 0.1, text, ha="center", va="bottom", fontsize=8, color=color)
    for b, x in [("amazon", 0.15), ("atlantic", 9.95)]:
        d, i = DATA[(b, "test")], median_forest_image(b)
        a = fig.add_axes((x / 12, 1.45 / 4.0, 1.9 / 12, 1.9 / 4.0)); a.imshow(rgb(d["x"][i])); a.set_xticks([]); a.set_yticks([])
        a2 = fig.add_axes(((x + 0.95) / 12, 0.4 / 4.0, 0.95 / 12, 0.95 / 4.0))
        a2.imshow(d["m"][i], cmap=ListedColormap(["#efe9d8", "#2e7d32"]), interpolation="nearest"); a2.set_xticks([]); a2.set_yticks([])
        n = {"amazon": "499 / 100 / 20", "atlantic": "485 / 100 / 20"}[b]
        ax.text(x + 0.95, 3.5, BIOMES[b].title(), ha="center", fontsize=11, weight="bold")
        ax.text(x + 0.45, 0.87, f"Sentinel-2\nR, G, B, NIR\n10 m, 512 px\ntrain/val/test\n{n}", ha="center", va="center", fontsize=7.5)
    box(2.95, 2.45, 2.55, 0.95, "Pretrained encoder\nResNet-34 (ImageNet)\nDINOv2 ViT-S | DINOv3 ViT-S\nDINOv3 ViT-L web | satellite", "#dbe8f7")
    box(2.95, 1.25, 2.55, 0.75, "U-Net decoder\n(ViT: 4-level UNETR pyramid)", "#f3f0e6")
    arrow(4.225, 2.45, 4.225, 2.0)
    box(5.95, 1.25, 2.45, 2.15, "Full (FT) or partial (PFT)\nfine-tuning\n\n40 epochs, 256 px crops\nBCE + Dice, AdamW, cosine\n\n5 seeds, paired by seed;\nHolm-corrected tests", "#f3f0e6")
    arrow(5.5, 2.3, 5.95, 2.3)
    arrow(2.1, 2.95, 2.95, 2.95, "train", color="#1971c2")
    arrow(8.4, 2.3, 9.9, 2.3, "test: official split\n+ whole biome (605)", color="#e8590c")
    ax.text(6.0, 0.3, "Each model is trained on one biome (checkpoint chosen on its validation split) and tested on both biomes;\n"
            "the reverse direction (Atlantic Forest \u2192 Amazon) is run as well.", ha="center", va="center", fontsize=8.5, style="italic")
    fig.savefig(out, dpi=dpi)
    print("saved", out)

draw_method(FIGS / "fig_method.png")